# Host mapping correction: ambiguous symbols withheld and full gene evidence preserved


66.04 first controlled partition: correct source-builder symbol mapping, preserve raw rhoptry gene evidence and verify full-source non-loss. Installed caches/tables and released calibration remain unchanged; migration is a separate partition. Compare the archived legacy parser on exactly the same reviewed files, preserving every ambiguity rather than guessing a protein.

In [1]:
import json, hashlib, gzip, tempfile, importlib.util
from pathlib import Path
import pandas as pd
from starplast import host as H, deposits as D, organisms as O
root=Path('/media/carruthers/mnt3/claude/toxoplasma_projects/datasets')
out=Path('results/host_symbol_mapping_2026_10_08')
legacy_path=Path('results/host_expression_source_review_2026_10_08/code/host.py')
spec=importlib.util.spec_from_file_location('starplast._legacy_symbol_mapping',legacy_path)
legacy=importlib.util.module_from_spec(spec)
spec.loader.exec_module(legacy)
inputs=[legacy_path,Path('starplast/host.py'),Path('starplast/deposits.py')]
reports={}
mappings={}
for species,names in H.UNIPROT_IDMAP.items():
    entries=root/H.UNIPROT_ROOT/names[1]
    inputs.extend(root/H.UNIPROT_ROOT/name for name in names)
    seen={}
    with gzip.open(entries,'rt') as stream:
        stream.readline()
        for line in stream:
            fields=line.rstrip('\n').split('\t')
            if len(fields)>1 and fields[1]: seen.setdefault(fields[1],set()).add(fields[0])
    ambiguous={k:sorted(v) for k,v in seen.items() if len(v)>1}
    previous=legacy.uniprot_index(str(root),species,log=lambda *a:None)
    current=H.uniprot_index(str(root),species)
    assert current['by_ensembl']==previous['by_ensembl']
    assert set(previous['by_symbol'])-set(current['by_symbol'])==set(ambiguous)
    assert all(current['by_symbol'][symbol]==previous['by_symbol'][symbol] for symbol in current['by_symbol'])
    assert all(symbol not in current['by_symbol'] for symbol in ambiguous)
    reports[species]={'reviewed_symbols':len(seen),'unambiguous_symbols':len(current['by_symbol']),'ambiguous_symbols':len(ambiguous),'ambiguities':ambiguous,'ensembl_mapping_unchanged':True}
    mappings[species]=current
(out/'mapping_review.json').write_text(json.dumps(reports,indent=2)+'\n')
print({k:{f:r[f] for f in ('reviewed_symbols','unambiguous_symbols','ambiguous_symbols','ensembl_mapping_unchanged')} for k,r in reports.items()})

uniprot index (human): 21,844 Ensembl genes, 20,132 symbols; 199 Ensembl ids and 58 symbols withheld for naming multiple reviewed proteins
uniprot index (mouse): 15,656 Ensembl genes, 16,920 symbols; 55 Ensembl ids and 35 symbols withheld for naming multiple reviewed proteins
{'human': {'reviewed_symbols': 20190, 'unambiguous_symbols': 20132, 'ambiguous_symbols': 58, 'ensembl_mapping_unchanged': True}, 'mouse': {'reviewed_symbols': 16955, 'unambiguous_symbols': 16920, 'ambiguous_symbols': 35, 'ensembl_mapping_unchanged': True}}


In [2]:
bindings_path=Path('results/source_recovery_final_2026_10_07_v2/bindings.json')
binding=json.loads(bindings_path.read_text())['host_k562_rhoptry_screen']
source=Path(binding['path'])
assert hashlib.sha256(source.read_bytes()).hexdigest()==binding['sha256']
inputs.extend((bindings_path,source))
with tempfile.TemporaryDirectory(prefix='starplast-rhoptry-symbol-review-') as temp:
    staging=Path(temp)
    reference=staging/H.UNIPROT_ROOT
    reference.parent.mkdir(parents=True)
    reference.symlink_to(root/H.UNIPROT_ROOT,target_is_directory=True)
    link=staging.joinpath(*D.K562)
    link.parent.mkdir(parents=True)
    link.symlink_to(source)
    evidence=D.k562_rhoptry_evidence(str(staging))
    corrected=D.k562_rhoptry_screen(str(staging))
raw=pd.read_excel(source,sheet_name='Rhoptry discharge MAgECK-MLE')
assert len(evidence)==len(raw)==20010
assert evidence.source_row.tolist()==list(range(2,len(raw)+2))
assert evidence.source_gene_symbol.tolist()==raw.Gene.tolist()
for field,column in [('rhoptry_discharge_score','Combined Rhoptry Score'),('rhoptry_discharge_beta','greenN|beta'),('rhoptry_discharge_fdr','greenN|wald-fdr')]:
    pd.testing.assert_series_equal(evidence[field],pd.to_numeric(raw[column],errors='coerce'),check_names=False)
symbols=evidence.gene_symbol
evidence['projection_status']=['ambiguous_reviewed_proteins' if s in reports['human']['ambiguities'] else 'unique_reviewed_protein' if s in mappings['human']['by_symbol'] else 'unmapped_reviewed_protein' for s in symbols]
evidence['proposed_host_id']=symbols.map(mappings['human']['by_symbol'])
evidence['candidate_accessions']=[reports['human']['ambiguities'].get(s,[]) for s in symbols]
evidence.to_parquet(out/'rhoptry_gene_evidence.parquet',index=False)
corrected.to_parquet(out/'rhoptry_corrected_projection_candidate.parquet',index=False)
assert evidence.projection_status.value_counts().to_dict()=={'unique_reviewed_protein':18700,'unmapped_reviewed_protein':1271,'ambiguous_reviewed_proteins':39}
assert not evidence.loc[evidence.projection_status=='ambiguous_reviewed_proteins','proposed_host_id'].notna().any()
assert len(corrected)==18700 and corrected.host_id.is_unique
installed_path=Path('starplast/data')/O.HOST_TABLES[O.HUMAN]
inputs.append(installed_path)
installed=pd.read_parquet(installed_path).set_index('host_id')
fields=['rhoptry_discharge_score','rhoptry_discharge_beta','rhoptry_discharge_fdr']
assert set(corrected.host_id).issubset(set(installed.index))
for field in fields:
    pd.testing.assert_series_equal(corrected.set_index('host_id')[field].sort_index(),installed.loc[corrected.host_id,field].sort_index(),check_dtype=False)
ambiguous=evidence[evidence.projection_status=='ambiguous_reviewed_proteins']
old_mapping=legacy.uniprot_index(str(root),'human',log=lambda *a:None)['by_symbol']
ambiguous=ambiguous.assign(legacy_assigned_host_id=ambiguous.gene_symbol.map(old_mapping))
ambiguous.to_parquet(out/'ambiguous_legacy_projection_review.parquet',index=False)
summary={'all_original_gene_rows_retained':len(evidence),'ambiguous_genes_retained_without_projection':len(ambiguous),'unmapped_genes_retained':1271,'corrected_unambiguous_candidate_projections':len(corrected),'all_three_fields_exact_on_unambiguous_installed_rows':True,'installed_table_changes':'none; migration and affected validation pending','evidence_scope':'K562 gene knockout; no gene score duplicated across ambiguous proteins'}
print(summary)

{'all_original_gene_rows_retained': 20010, 'ambiguous_genes_retained_without_projection': 39, 'unmapped_genes_retained': 1271, 'corrected_unambiguous_candidate_projections': 18700, 'all_three_fields_exact_on_unambiguous_installed_rows': True, 'installed_table_changes': 'none; migration and affected validation pending', 'evidence_scope': 'K562 gene knockout; no gene score duplicated across ambiguous proteins'}


In [3]:
def sha(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1<<20),b''):digest.update(block)
    return digest.hexdigest()
(out/'manifest.json').write_text(json.dumps({'summary':summary,'input_sha256':{str(p.resolve()):sha(p) for p in inputs},'outputs':{p.name:sha(p) for p in out.iterdir() if p.is_file()}},indent=2)+'\n')
summary

{'all_original_gene_rows_retained': 20010, 'ambiguous_genes_retained_without_projection': 39, 'unmapped_genes_retained': 1271, 'corrected_unambiguous_candidate_projections': 18700, 'all_three_fields_exact_on_unambiguous_installed_rows': True, 'installed_table_changes': 'none; migration and affected validation pending', 'evidence_scope': 'K562 gene knockout; no gene score duplicated across ambiguous proteins'}

This corrects the future source projection only. Existing wrong protein projections still need an explicit installed-table migration; no completion tick is earned yet. Preserve all original gene-level values and mapping alternatives. No aliases or full gene identifiers inferred from symbols. All Ensembl mappings are unchanged in this partition, including the separately recorded PAR_Y/Atlas-cutoff follow-up. No biological accuracy or inference-pack completion claim.